In [2]:
import pandas as pd
from bs4.builder import FAST

books = pd.read_csv("books_with_the_categories.csv")

In [3]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k=None,
                      device=0)
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419787786900997},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [4]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [5]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.654840350151062},
  {'label': 'neutral', 'score': 0.1698523312807083},
  {'label': 'sadness', 'score': 0.11640936881303787},
  {'label': 'surprise', 'score': 0.020700648427009583},
  {'label': 'disgust', 'score': 0.01910071074962616},
  {'label': 'joy', 'score': 0.015161431394517422},
  {'label': 'anger', 'score': 0.003935148473829031}]]

In [6]:
classifier(books["description"][0].split("."))

[[{'label': 'surprise', 'score': 0.7296019792556763},
  {'label': 'neutral', 'score': 0.14038611948490143},
  {'label': 'fear', 'score': 0.06816229224205017},
  {'label': 'joy', 'score': 0.04794260114431381},
  {'label': 'anger', 'score': 0.009156369604170322},
  {'label': 'disgust', 'score': 0.0026284789200872183},
  {'label': 'sadness', 'score': 0.002122164238244295}],
 [{'label': 'neutral', 'score': 0.4493708610534668},
  {'label': 'disgust', 'score': 0.2735908627510071},
  {'label': 'joy', 'score': 0.10908335447311401},
  {'label': 'sadness', 'score': 0.09362750500440598},
  {'label': 'anger', 'score': 0.04047826677560806},
  {'label': 'surprise', 'score': 0.026970183476805687},
  {'label': 'fear', 'score': 0.006879056338220835}],
 [{'label': 'neutral', 'score': 0.6462154388427734},
  {'label': 'sadness', 'score': 0.24273377656936646},
  {'label': 'disgust', 'score': 0.04342271387577057},
  {'label': 'surprise', 'score': 0.028300553560256958},
  {'label': 'joy', 'score': 0.01421149

In [7]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [8]:
sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives'

In [9]:
predictions[0]

[{'label': 'surprise', 'score': 0.7296019792556763},
 {'label': 'neutral', 'score': 0.14038611948490143},
 {'label': 'fear', 'score': 0.06816229224205017},
 {'label': 'joy', 'score': 0.04794260114431381},
 {'label': 'anger', 'score': 0.009156369604170322},
 {'label': 'disgust', 'score': 0.0026284789200872183},
 {'label': 'sadness', 'score': 0.002122164238244295}]

In [10]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009156369604170322},
 {'label': 'disgust', 'score': 0.0026284789200872183},
 {'label': 'fear', 'score': 0.06816229224205017},
 {'label': 'joy', 'score': 0.04794260114431381},
 {'label': 'neutral', 'score': 0.14038611948490143},
 {'label': 'sadness', 'score': 0.002122164238244295},
 {'label': 'surprise', 'score': 0.7296019792556763}]

In [14]:
import numpy as np

emotion_labels = ["anger","disgust","fear","joy","sadness","surprise","neutral"]
isbn=[]
emotion_scores = {label:[] for label in emotion_labels}

def calculate_max_emotion_scores (predictions):
    per_emotion_scores = {label:[] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
        return{label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [15]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [16]:
emotion_scores

{'anger': [np.float64(0.009156369604170322),
  np.float64(0.005966846831142902),
  np.float64(0.04130091518163681),
  np.float64(0.01603621616959572),
  np.float64(0.013624406419694424),
  np.float64(0.009158514440059662),
  np.float64(0.003973887301981449),
  np.float64(0.023656276986002922),
  np.float64(0.008916167542338371),
  np.float64(0.30066978931427)],
 'disgust': [np.float64(0.0026284789200872183),
  np.float64(0.002887063194066286),
  np.float64(0.024568339809775352),
  np.float64(0.0606951043009758),
  np.float64(0.12224280089139938),
  np.float64(0.012228727340698242),
  np.float64(0.003816467011347413),
  np.float64(0.009091970510780811),
  np.float64(0.005425345618277788),
  np.float64(0.27948111295700073)],
 'fear': [np.float64(0.06816229224205017),
  np.float64(0.003809873480349779),
  np.float64(0.10406076163053513),
  np.float64(0.0016916368622332811),
  np.float64(0.0950433760881424),
  np.float64(0.036795180290937424),
  np.float64(0.0012676790356636047),
  np.floa

In [17]:
from tqdm import tqdm

emotion_labels = ["anger","disgust","fear","joy","sadness","surprise","neutral"]
isbn=[]
emotion_scores = {label:[] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 6057/6057 [19:09<00:00,  5.27it/s] 


In [18]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"]=isbn

In [19]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.009156,0.002628,0.068162,0.047943,0.140386,0.002122,0.729602,9780002005883
1,0.005967,0.002887,0.003810,0.704422,0.217759,0.004509,0.060646,9780002261982
2,0.041301,0.024568,0.104061,0.767238,0.042176,0.010860,0.009796,9780006178736
3,0.016036,0.060695,0.001692,0.161757,0.732685,0.020988,0.006147,9780006280897
4,0.013624,0.122243,0.095043,0.008336,0.272614,0.475880,0.012260,9780006280934
...,...,...,...,...,...,...,...,...
6052,0.005602,0.003775,0.018216,0.400263,0.338892,0.005487,0.227765,9788173031014
6053,0.008463,0.009146,0.013295,0.620454,0.329752,0.010788,0.008101,9788179921623
6054,0.005475,0.034544,0.003970,0.258354,0.648010,0.017372,0.032275,9788185300535
6055,0.002837,0.003137,0.001166,0.958549,0.028252,0.002916,0.003142,9789027712059


In [20]:
books = pd.merge(books,emotions_df, on="isbn13")

In [21]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.009156,0.002628,0.068162,0.047943,0.140386,0.002122,0.729602
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.005967,0.002887,0.003810,0.704422,0.217759,0.004509,0.060646
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.041301,0.024568,0.104061,0.767238,0.042176,0.010860,0.009796
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.016036,0.060695,0.001692,0.161757,0.732685,0.020988,0.006147
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.013624,0.122243,0.095043,0.008336,0.272614,0.475880,0.012260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6052,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.005602,0.003775,0.018216,0.400263,0.338892,0.005487,0.227765
6053,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.008463,0.009146,0.013295,0.620454,0.329752,0.010788,0.008101
6054,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.005475,0.034544,0.003970,0.258354,0.648010,0.017372,0.032275
6055,9789027712059,9027712050,The Berlin Phenomenology,Georg Wilhelm Friedrich Hegel,History,http://books.google.com/books/content?id=Vy7Sk...,Since the three volume edition ofHegel's Philo...,1981.0,0.00,210.0,...,The Berlin Phenomenology,9789027712059 Since the three volume edition o...,Nonfiction,0.002837,0.003137,0.001166,0.958549,0.028252,0.002916,0.003142


In [22]:
books.to_csv("books_with_emotions.csv", index=False)